# M5 · Assistant en derja — classifieur d'intentions

**Tâche** : comprendre une question d'agriculteur écrite en derja (arabizi ou écriture arabe) ou en français, et
détecter l'intention pour router vers le bon module (M1 maladie, M2 irrigation / météo, M3 récolte, M4 prix,
M6 comptage, conseil général, salutation, hors sujet).

| | |
|---|---|
| Données | `data/m5_intents.csv` : questions écrites par l'équipe (9 intentions, arabizi + arabe + français) · test indépendant : `ai-services/m5-assistant/tests/data/m5_probe_set.csv` (25 questions jamais vues) |
| Prétraitement | `normalize()` du service (minuscules, alef / ta marbouta unifiés, répétitions réduites…) : **identique** à l'inférence |
| Baseline | TF-IDF mots (1–2) + régression logistique (le plan) |
| Modèles | TF-IDF caractères (2–5, robuste aux orthographes arabizi) · union mots + caractères |
| Métriques | F1 macro (validation croisée stratifiée 5 plis + test 20 %), justesse sur le probe set |
| Export | `models/m5_ensemble/` : `tfidf.joblib` + `config.json` (`weight_setfit = 0` : modèle TF-IDF seul) |

In [ ]:
%matplotlib inline
import json
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import FeatureUnion, make_pipeline

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "ai-services" / "m5-assistant"))
from app.preprocessing import normalize  # the exact preprocessing of the service

SEED = 42
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = ROOT / "models" / "m5_ensemble"
print("python", platform.python_version(), "| scikit-learn", sklearn.__version__)

## 1 · Données

In [ ]:
data = pd.read_csv("data/m5_intents.csv")
probe = pd.read_csv(ROOT / "ai-services/m5-assistant/tests/data/m5_probe_set.csv")
data["clean"] = data["text"].map(normalize)
overlap = set(data["clean"]) & set(probe["text"].map(normalize))
print(f"{len(data)} questions, {data['intent'].nunique()} intentions · probe set : {len(probe)} · recouvrement : {len(overlap)}")
script = np.where(data["text"].str.contains("[\u0600-\u06FF]"), "arabe", "latin (arabizi / français)")
pd.crosstab(data["intent"], script, margins=True)

## 2 · Comparaison des modèles

In [ ]:
def word_model():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                         LogisticRegression(C=10, max_iter=2000))

def char_model():
    return make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
                         LogisticRegression(C=10, max_iter=2000))

def union_model():
    features = FeatureUnion([
        ("words", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
    ])
    return make_pipeline(features, LogisticRegression(C=10, max_iter=2000))

candidates = {"TF-IDF mots + LR (baseline)": word_model, "TF-IDF caractères + LR": char_model,
              "TF-IDF mots + caractères + LR": union_model}
x_train, x_test, y_train, y_test = train_test_split(data["clean"], data["intent"], test_size=0.2,
                                                    stratify=data["intent"], random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = {}
for name, factory in candidates.items():
    cv_f1 = cross_val_score(factory(), data["clean"], data["intent"], cv=cv, scoring="f1_macro")
    model = factory().fit(x_train, y_train)
    full = factory().fit(data["clean"], data["intent"])
    rows[name] = {
        "F1 macro CV (moy.)": cv_f1.mean(), "F1 macro CV (écart-type)": cv_f1.std(),
        "F1 macro test 20 %": f1_score(y_test, model.predict(x_test), average="macro"),
        "probe set (/25)": int((full.predict(probe["text"].map(normalize)) == probe["intent"]).sum()),
    }
results = pd.DataFrame(rows).T
results.to_csv(OUTPUT_DIR / "results.csv")
results.round(3)

In [ ]:
best_name = results["F1 macro CV (moy.)"].idxmax()
best = candidates[best_name]().fit(x_train, y_train)
print("meilleur :", best_name)
print(classification_report(y_test, best.predict(x_test), digits=3))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, best.predict(x_test), ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title("Matrice de confusion (test 20 %)")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "confusion.png", dpi=110); plt.show()

## 3 · Seuil de clarification

Avec un seul modèle, le service pose la question de clarification quand la **confiance maximale** est sous le seuil.
On choisit le seuil sur les prédictions hors-plis de la validation croisée : le plus petit qui garde ≥ 85 % de bonnes
réponses parmi les questions auxquelles l'assistant répond directement.

In [ ]:
from sklearn.model_selection import cross_val_predict

proba = cross_val_predict(candidates[best_name](), data["clean"], data["intent"], cv=cv, method="predict_proba")
labels = sorted(data["intent"].unique())
conf, pred = proba.max(axis=1), np.array(labels)[proba.argmax(axis=1)]
rows = []
for t in np.arange(0.20, 0.71, 0.05):
    kept = conf >= t
    rows.append({"seuil": round(float(t), 2), "% répondu direct": 100 * kept.mean(),
                 "justesse des réponses directes": 100 * (pred[kept] == data["intent"].to_numpy()[kept]).mean()})
thresholds = pd.DataFrame(rows)
ok = thresholds[thresholds["justesse des réponses directes"] >= 85]
threshold = float(ok["seuil"].min()) if len(ok) else 0.5
print("seuil retenu :", threshold)
thresholds.round(1)

## 4 · Export (modèle entraîné sur toutes les questions)

In [ ]:
import joblib

final = candidates[best_name]().fit(data["clean"], data["intent"])
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final, MODEL_DIR / "tfidf.joblib")
config = {
    "labels": [str(c) for c in final.classes_],
    "weight_setfit": 0.0,
    "clarify_threshold": threshold,
    "model": best_name,
    "sklearn_version": sklearn.__version__,
    "trained_on": len(data),
    "metrics": json.loads(results.round(4).to_json(orient="index")),
}
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print("probe set avec le modèle final :", int((final.predict(probe["text"].map(normalize)) == probe["intent"]).sum()), "/ 25")
sorted(p.name for p in MODEL_DIR.iterdir())